In [ ]:
# =============================================================================
# PALM Water Quality Modeling - Chl-a - 2015-2016
# =============================================================================
#
# Author: Juan Gabriel Loaiza
# Affiliation: Universidad Autónoma de Sinaloa (UAS), Mexico
# Project: Land-use/land-cover and water-quality analysis of the
#          Adolfo Lopez Mateos Reservoir (PALM), Sinaloa, Mexico
# Repository: PALM-LULC-Water-Quality
# Repository version: v1.0.0
#
# Description:
# Water-quality modeling workflow using Landsat 8 spectral reflectance data.
# The analytical/modeling workflow is preserved from the manuscript analysis;
# repository-relative paths replace the original private Google Drive paths.
#
# License: See LICENSE in the repository root.
# =============================================================================

# ============================================================
# Box-Cox Transformation and Multiple Linear Regression
# Water Quality Modeling (TDS/SDT) using Spectral Predictors
# ============================================================
#
# Description:
# This script implements a Box-Cox transformation combined with
# multiple linear regression (OLS) to model Total Organic Carbon
# (TDS/SDT) from remote sensing spectral bands.
#
# Main Features:
# 1. Optimizes the Box-Cox parameter (λ1) by minimizing MSE.
# 2. Applies the standardized Box-Cox transformation.
# 3. Fits an Ordinary Least Squares (OLS) regression model.
# 4. Back-transforms predictions to the original response scale.
# 5. Computes performance metrics and diagnostic statistics.
# 6. Identifies outliers and influential observations.
# 7. Generates diagnostic plots.
# 8. Exports results and statistics to Excel.
#
# Outputs:
# - Regression coefficients
# - ANOVA table
# - Model performance metrics
# - Prediction intervals
# - Diagnostic plots
# - Excel report
#
# Author: Juan G. Loaiza
# Institution: Universidad Autónoma de Sinaloa (UAS)
# ============================================================


# ── 1. LIBRARIES ────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import statsmodels.api as sm

from scipy.stats import skew, kurtosis
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import OLSInfluence
from sklearn.metrics import r2_score, mean_squared_error


# ── GRAPH FONT SIZES ─────────────────────────────────────────

plt.rcParams.update({
    'font.size': 15,
    'axes.titlesize': 17,
    'axes.labelsize': 15,
    'xtick.labelsize': 15,
    'ytick.labelsize': 15,
    'legend.fontsize': 15,
})


# ── 2. DATA LOADING ─────────────────────────────────────────

# ── REPOSITORY PATHS ────────────────────────────────────────
# Works when the notebook is run from its repository folder or a descendant.
from pathlib import Path

def find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'README.md').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Run this notebook from within the "
        "PALM-LULC-Water-Quality repository."
    )

REPO_ROOT = find_repo_root()
MODEL_RESULTS_DIR = REPO_ROOT / 'results' / 'model_performance'
PREDICTIONS_DIR = REPO_ROOT / 'derived_data' / 'water_quality_predictions'
FIGURES_DIR = REPO_ROOT / 'results' / 'figures' / 'water_quality_models'
for _dir in (MODEL_RESULTS_DIR, PREDICTIONS_DIR, FIGURES_DIR):
    _dir.mkdir(parents=True, exist_ok=True)

# CARPETA_SALIDA is retained as an alias so the original plotting/model code
# remains unchanged. Model tables and figures are written below using dedicated
# repository directories where explicitly defined.
CARPETA_SALIDA = str(MODEL_RESULTS_DIR)

RUTA = str(REPO_ROOT / 'data' / 'water_quality' / 'Water_quality_and_reflectance_model_inputs.xlsx')

HOJA         = 'Cha-a_2015-2016'
COL_Y        = 'Cha-a'
LAMBDA2      = 0.0
LAMBDA_RANGO = (-2.0, 2.0, 0.01)
ALPHA        = 0.05


# ── AUTOMATIC PERIOD FOR FIGURE TITLES ──────────────────────
# Example:
# Cha-a_2013-2014 → 2013–2014
# Cha-a_2015-2016 → 2015–2016
# Cha-a_2017-2018 → 2017–2018

PERIODO = HOJA.split('_')[-1].replace('-', '–')

print(f"Period: {PERIODO}")


df = pd.read_excel(RUTA, sheet_name=HOJA)

print(f"Sheet: {HOJA} | Dimensions: {df.shape}")
print("Columns:", df.columns.tolist())

cols_x = [
    c for c in df.columns
    if str(c).upper().startswith('B')
]

print(f"\nDependent variable  : {COL_Y}")
print(f"Predictor variables : {cols_x}")


# ── 3. CLEANING ─────────────────────────────────────────────

datos = (
    df[[COL_Y] + cols_x]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .reset_index(drop=True)
    .astype(float)
)

print(f"\nValid rows: {len(datos)}")

y = datos[COL_Y].values
X = datos[cols_x].values

n, k = X.shape


if np.any(y + LAMBDA2 <= 0):

    raise ValueError(
        f"Values Y + lambda2 <= 0 found. "
        f"Adjust LAMBDA2 (currently {LAMBDA2}) or review the data."
    )


# ── 4. BOX-COX FUNCTIONS ────────────────────────────────────

def calcular_k2(
    y: np.ndarray,
    lambda2: float = 0.0
) -> float:

    """Geometric mean of (Y + lambda2)."""

    return np.exp(
        np.mean(
            np.log(y + lambda2)
        )
    )


def boxcox_transform(
    y: np.ndarray,
    lambda1: float,
    lambda2: float,
    K2: float
) -> np.ndarray:

    """
    Standardized Box-Cox transformation (StatGraphics style):

    lambda1 != 0 →
    W = ((Y+λ2)^λ1 − 1) / (λ1 · K2^(λ1−1))

    lambda1 == 0 →
    W = K2 · ln(Y+λ2)
    """

    yp = y + lambda2

    if np.isclose(lambda1, 0.0):

        return K2 * np.log(yp)

    return (
        (yp ** lambda1 - 1.0)
        /
        (lambda1 * K2 ** (lambda1 - 1.0))
    )


def boxcox_inverse(
    W: np.ndarray,
    lambda1: float,
    lambda2: float,
    K2: float
) -> np.ndarray:

    """
    Inverse transformation back to Y scale.

    Positions where base <= 0 return NaN
    (may occur at some prediction interval limits).
    """

    W = np.asarray(
        W,
        dtype=float
    )

    if np.isclose(lambda1, 0.0):

        return (
            np.exp(W / K2)
            - lambda2
        )

    base = (
        1.0
        + W
        * lambda1
        * K2 ** (lambda1 - 1.0)
    )

    yp = np.where(
        base > 0,
        base ** (1.0 / lambda1),
        np.nan
    )

    return yp - lambda2


# ── 5. LAMBDA1 OPTIMIZATION ─────────────────────────────────

lam_min, lam_max, lam_step = LAMBDA_RANGO

K2 = calcular_k2(
    y,
    LAMBDA2
)

X_const = sm.add_constant(
    X,
    has_constant='add'
)


records = []


for lam in np.arange(
    lam_min,
    lam_max + lam_step,
    lam_step
):

    try:

        W_tmp = boxcox_transform(
            y,
            lam,
            LAMBDA2,
            K2
        )

        mod_tmp = sm.OLS(
            W_tmp,
            X_const
        ).fit()

        resid = (
            W_tmp
            - np.asarray(
                mod_tmp.fittedvalues
            )
        )

        records.append({

            'lambda1':
                lam,

            'MSE_W':
                np.mean(
                    resid ** 2
                ),

            'Skewness_W':
                skew(
                    W_tmp,
                    bias=False
                ),

            'Kurtosis_W':
                kurtosis(
                    W_tmp,
                    fisher=True,
                    bias=False
                ),
        })

    except Exception:

        continue


if not records:

    raise ValueError(
        "No valid lambda could be evaluated."
    )


tabla_lambda = (
    pd.DataFrame(records)
    .sort_values('lambda1')
    .reset_index(drop=True)
)


lambda_opt = tabla_lambda.loc[
    tabla_lambda['MSE_W'].idxmin(),
    'lambda1'
]


print(
    f"\nOptimal Lambda1: "
    f"{lambda_opt:.4f}"
)


# ── 6. FINAL FIT ────────────────────────────────────────────

W = boxcox_transform(
    y,
    lambda_opt,
    LAMBDA2,
    K2
)


modelo = sm.OLS(
    W,
    X_const
).fit()


W_hat = np.asarray(
    modelo.fittedvalues
)


resid_W = (
    W - W_hat
)


y_hat = boxcox_inverse(
    W_hat,
    lambda_opt,
    LAMBDA2,
    K2
)


resid_Y = (
    y - y_hat
)


# ── 7. METRICS ──────────────────────────────────────────────

metricas = {

    'lambda1_optimal':
        lambda_opt,

    'lambda2':
        LAMBDA2,

    'K2':
        K2,

    'R2_W':
        modelo.rsquared,

    'R2_W_adjusted':
        modelo.rsquared_adj,

    'MSE_W':
        float(
            np.mean(
                resid_W ** 2
            )
        ),

    'RMSE_W':
        float(
            np.sqrt(
                np.mean(
                    resid_W ** 2
                )
            )
        ),

    'MAE_W':
        float(
            np.mean(
                np.abs(
                    resid_W
                )
            )
        ),

    'Durbin_Watson':
        float(
            durbin_watson(
                modelo.resid
            )
        ),

    'R2_Y':
        r2_score(
            y,
            y_hat
        ),

    'MSE_Y':
        mean_squared_error(
            y,
            y_hat
        ),

    'RMSE_Y':
        float(
            np.sqrt(
                mean_squared_error(
                    y,
                    y_hat
                )
            )
        ),

    'MAE_Y':
        float(
            np.mean(
                np.abs(
                    resid_Y
                )
            )
        ),
}


tabla_metricas = pd.DataFrame(
    metricas.items(),
    columns=[
        'Metric',
        'Value'
    ]
)


# ── 8. COEFFICIENTS ─────────────────────────────────────────

tabla_coef = pd.DataFrame({

    'Variable':
        ['Intercept'] + cols_x,

    'Coefficient':
        modelo.params,

    'Std_Error':
        modelo.bse,

    't_value':
        modelo.tvalues,

    'p_value':
        modelo.pvalues,
})


terminos = [
    f"{modelo.params[0]:.6f}"
]


for nombre, coef in zip(
    cols_x,
    modelo.params[1:]
):

    terminos.append(
        f"{'+ ' if coef >= 0 else '- '}"
        f"{abs(coef):.6f}·{nombre}"
    )


print(
    f"\nW = {' '.join(terminos)}"
)


if np.isclose(
    lambda_opt,
    0.0
):

    print(
        f"Transformation: "
        f"W = {K2:.6f} · "
        f"ln(Y + {LAMBDA2})"
    )

else:

    print(
        f"Transformation: "
        f"W = ((Y+{LAMBDA2})^"
        f"{lambda_opt:.4f} − 1) "
        f"/ ({lambda_opt:.4f} · "
        f"{K2:.6f}^"
        f"({lambda_opt:.4f}−1))"
    )


# ── 9. ANOVA ────────────────────────────────────────────────

ss_total = np.sum(
    (W - W.mean()) ** 2
)

ss_resid = np.sum(
    resid_W ** 2
)

ss_model = (
    ss_total
    - ss_resid
)

df_model = k

df_resid = (
    n - k - 1
)

ms_model = (
    ss_model
    / df_model
)

ms_resid = (
    ss_resid
    / df_resid
)


anova = pd.DataFrame({

    'Source': [
        'Model',
        'Residual',
        'Total (Corr.)'
    ],

    'SS': [
        ss_model,
        ss_resid,
        ss_total
    ],

    'df': [
        df_model,
        df_resid,
        n - 1
    ],

    'MS': [
        ms_model,
        ms_resid,
        np.nan
    ],

    'F': [
        ms_model / ms_resid,
        np.nan,
        np.nan
    ],

    'p_value': [
        modelo.f_pvalue,
        np.nan,
        np.nan
    ],
})


# ── 10. DIAGNOSTICS: INFLUENCE, OUTLIERS, LEVERAGE ──────────

influence = OLSInfluence(
    modelo
)

resid_student = (
    influence
    .resid_studentized_external
)

leverage = (
    influence
    .hat_matrix_diag
)

cooks_d = (
    influence
    .cooks_distance[0]
)

umbral_lev = (
    3 * (k + 1)
    / n
)


tabla_diag = datos.assign(

    W_observed=W,

    W_estimated=W_hat,

    Y_estimated=y_hat,

    Residual_W=resid_W,

    Residual_Y=resid_Y,

    Studentized_Residual=
        resid_student,

    Leverage=
        leverage,

    Cooks_D=
        cooks_d,
)


tabla_outliers = (
    tabla_diag.loc[
        np.abs(
            resid_student
        ) > 2
    ].copy()
)


tabla_influyentes = (
    tabla_diag.loc[
        leverage
        >= umbral_lev
    ].copy()
)


# ── 11. PREDICTION INTERVALS (Y scale) ──────────────────────

pred_df = (
    modelo
    .get_prediction(
        X_const
    )
    .summary_frame(
        alpha=ALPHA
    )
)


tabla_pred = pd.DataFrame({

    'Y_estimated':
        boxcox_inverse(
            pred_df['mean'].values,
            lambda_opt,
            LAMBDA2,
            K2
        ),

    'Y_CI_lower':
        boxcox_inverse(
            pred_df['mean_ci_lower'].values,
            lambda_opt,
            LAMBDA2,
            K2
        ),

    'Y_CI_upper':
        boxcox_inverse(
            pred_df['mean_ci_upper'].values,
            lambda_opt,
            LAMBDA2,
            K2
        ),

    'Y_PI_lower':
        boxcox_inverse(
            pred_df['obs_ci_lower'].values,
            lambda_opt,
            LAMBDA2,
            K2
        ),

    'Y_PI_upper':
        boxcox_inverse(
            pred_df['obs_ci_upper'].values,
            lambda_opt,
            LAMBDA2,
            K2
        ),
})


nans_por_col = (
    tabla_pred
    .isna()
    .sum()
)


if nans_por_col.any():

    print(
        "\nNaN in intervals "
        "(base <= 0 when inverting Box-Cox):"
    )

    print(
        nans_por_col[
            nans_por_col > 0
        ]
    )


tabla_final = pd.concat(
    [
        tabla_diag,
        tabla_pred
    ],
    axis=1
)


# ── 12. PLOTS (one figure per plot) ─────────────────────────

def guardar_fig(
    fig,
    nombre
):

    ruta = (
        f'{CARPETA_SALIDA}/'
        f'{nombre}_{COL_Y}_{HOJA}.png'
    )

    fig.savefig(
        ruta,
        dpi=150,
        bbox_inches='tight'
    )

    print(
        f"  Saved: {ruta}"
    )

    plt.show()

    plt.close(fig)


# ============================================================
# a) MSE vs lambda1
# ============================================================

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.plot(
    tabla_lambda['lambda1'],
    tabla_lambda['MSE_W']
)

ax.axvline(
    lambda_opt,
    ls='--',
    label=f'λ={lambda_opt:.2f}'
)

ax.set(
    title=f'MSE vs λ1 ({PERIODO})',
    xlabel='λ1',
    ylabel='MSE (W)'
)

ax.legend()

ax.grid(
    alpha=0.3
)

fig.tight_layout()

guardar_fig(
    fig,
    '01_MSE_vs_lambda'
)


# ============================================================
# b) Skewness and Kurtosis vs lambda1
# ============================================================

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.plot(
    tabla_lambda['lambda1'],
    tabla_lambda['Skewness_W'],
    label='Skewness'
)

ax.plot(
    tabla_lambda['lambda1'],
    tabla_lambda['Kurtosis_W'],
    label='Kurtosis'
)

ax.axhline(
    2,
    ls='--',
    color='gray'
)

ax.axhline(
    -2,
    ls='--',
    color='gray'
)

ax.axvline(
    lambda_opt,
    ls='--',
    label=f'λ opt={lambda_opt:.2f}'
)

ax.set(
    title=(
        f'Skewness and Kurtosis vs λ1 '
        f'({PERIODO})'
    ),
    xlabel='λ1',
    ylabel='Value'
)

ax.legend()

ax.grid(
    alpha=0.3
)

fig.tight_layout()

guardar_fig(
    fig,
    '02_Skewness_Kurtosis_vs_lambda'
)


# ============================================================
# c) Observed vs Estimated (Y)
# ============================================================

mn = min(
    y.min(),
    y_hat.min()
)

mx = max(
    y.max(),
    y_hat.max()
)


fig, ax = plt.subplots(
    figsize=(6, 6)
)

ax.scatter(
    y,
    y_hat,
    alpha=0.7
)

ax.plot(
    [mn, mx],
    [mn, mx],
    'r-'
)

ax.set(
    title=(
        f'Observed vs Estimated '
        f'({COL_Y}, {PERIODO})'
    ),
    xlabel='Observed',
    ylabel='Estimated'
)


ax.text(
    0.05,
    0.95,
    (
        f"R²={metricas['R2_Y']:.4f}\n"
        f"RMSE={metricas['RMSE_Y']:.4f}"
    ),
    transform=ax.transAxes,
    va='top',
    bbox=dict(
        facecolor='white',
        alpha=0.8
    )
)

ax.grid(
    alpha=0.3
)

fig.tight_layout()

guardar_fig(
    fig,
    '03_Observed_vs_Estimated'
)


# ============================================================
# d) Studentized Residuals vs Estimated Y
# ============================================================

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.scatter(
    y_hat,
    resid_student,
    alpha=0.7
)


for lim in (
    2,
    -2
):

    ax.axhline(
        lim,
        ls='--',
        color='gray'
    )


ax.axhline(
    0,
    color='black'
)


ax.set(
    title=(
        f'Studentized Residuals vs Estimated '
        f'({COL_Y}, {PERIODO})'
    ),
    xlabel=f'Estimated {COL_Y}',
    ylabel='Studentized Residual'
)


ax.grid(
    alpha=0.3
)

fig.tight_layout()

guardar_fig(
    fig,
    '04_Residuals_vs_Estimated_Y'
)


# ============================================================
# e) Studentized Residuals vs Observation Number
# ============================================================

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.scatter(
    np.arange(
        1,
        n + 1
    ),
    resid_student,
    alpha=0.7
)


for lim in (
    2,
    -2
):

    ax.axhline(
        lim,
        ls='--',
        color='gray'
    )


ax.axhline(
    0,
    color='black'
)


ax.set(
    title=(
        f'Studentized Residuals vs '
        f'Observation Number ({PERIODO})'
    ),
    xlabel='Observation Number',
    ylabel='Studentized Residual'
)


ax.grid(
    alpha=0.3
)

fig.tight_layout()

guardar_fig(
    fig,
    '05_Residuals_vs_Observation'
)


# ============================================================
# f) Studentized Residuals vs Estimated W
# ============================================================

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.scatter(
    W_hat,
    resid_student,
    alpha=0.7
)


for lim in (
    2,
    -2
):

    ax.axhline(
        lim,
        ls='--',
        color='gray'
    )


ax.axhline(
    0,
    color='black'
)


ax.set(
    title=(
        f'Studentized Residuals vs '
        f'Estimated W ({PERIODO})'
    ),
    xlabel='Estimated W',
    ylabel='Studentized Residual'
)


ax.grid(
    alpha=0.3
)

fig.tight_layout()

guardar_fig(
    fig,
    '06_Residuals_vs_Estimated_W'
)


# ── 13. CONSOLE SUMMARY ─────────────────────────────────────

print(
    "\n" +
    "=" * 55
)

print(
    "GENERAL METRICS"
)

print(
    "=" * 55
)

print(
    tabla_metricas.to_string(
        index=False
    )
)


print(
    "\n" +
    "=" * 55
)

print(
    "ANOVA"
)

print(
    "=" * 55
)

print(
    anova.to_string(
        index=False
    )
)


print(
    "\n" +
    "=" * 55
)

print(
    "COEFFICIENTS"
)

print(
    "=" * 55
)

print(
    tabla_coef.to_string(
        index=False
    )
)


print(
    "\n" +
    "=" * 55
)

print(
    f"OUTLIERS "
    f"(|studentized residual| > 2) "
    f"– {len(tabla_outliers)} found"
)

print(
    "=" * 55
)


if tabla_outliers.empty:

    print(
        "None detected."
    )

else:

    print(
        tabla_outliers[
            [
                COL_Y,
                'Y_estimated',
                'Residual_Y',
                'Studentized_Residual'
            ]
        ].to_string()
    )


print(
    "\n" +
    "=" * 55
)

print(
    f"INFLUENTIAL POINTS "
    f"(leverage >= {umbral_lev:.4f}) "
    f"– {len(tabla_influyentes)} found"
)

print(
    "=" * 55
)


if tabla_influyentes.empty:

    print(
        "None detected."
    )

else:

    print(
        tabla_influyentes[
            [
                COL_Y,
                'Y_estimated',
                'Studentized_Residual',
                'Leverage',
                'Cooks_D'
            ]
        ].to_string()
    )


print(
    "\n"
    + modelo.summary().as_text()
)


# ── OBSERVED VS ESTIMATED TABLE ─────────────────────────────

tabla_obs_est = pd.DataFrame({

    'Observed':
        y,

    'Estimated':
        y_hat,

    'Residual':
        resid_Y,

    'Absolute_Residual':
        np.abs(
            resid_Y
        ),

    'Squared_Residual':
        resid_Y ** 2,

}).round(6)


metricas_obs_est = pd.DataFrame({

    'Metric': [

        'R2_observed_vs_estimated',

        'RMSE_observed_vs_estimated',

        'MSE_observed_vs_estimated',

        'MAE_observed_vs_estimated',
    ],

    'Value': [

        metricas['R2_Y'],

        metricas['RMSE_Y'],

        metricas['MSE_Y'],

        metricas['MAE_Y']
    ],
})


print(
    "\n"
    + "=" * 55
)

print(
    "OBSERVED VS ESTIMATED TABLE"
)

print(
    "=" * 55
)

print(
    tabla_obs_est.to_string()
)


print(
    "\n"
    + "=" * 55
)

print(
    "OBSERVED VS ESTIMATED METRICS"
)

print(
    "=" * 55
)

print(
    metricas_obs_est.to_string(
        index=False
    )
)


# ── 14. EXPORT TO EXCEL ─────────────────────────────────────

SALIDA = (
    f'{CARPETA_SALIDA}/'
    f'Resultados_BoxCox_{COL_Y}_{HOJA}.xlsx'
)


with pd.ExcelWriter(
    SALIDA,
    engine='openpyxl'
) as writer:

    datos.to_excel(
        writer,
        sheet_name='Data_Used',
        index=False
    )

    tabla_coef.to_excel(
        writer,
        sheet_name='Coefficients',
        index=False
    )

    anova.to_excel(
        writer,
        sheet_name='ANOVA',
        index=False
    )

    tabla_metricas.to_excel(
        writer,
        sheet_name='Metrics',
        index=False
    )

    tabla_lambda.to_excel(
        writer,
        sheet_name='MSE_vs_Lambda',
        index=False
    )

    tabla_final.to_excel(
        writer,
        sheet_name='Results',
        index=False
    )

    tabla_outliers.to_excel(
        writer,
        sheet_name='Outliers',
        index=False
    )

    tabla_influyentes.to_excel(
        writer,
        sheet_name='Influential',
        index=False
    )

    tabla_obs_est.to_excel(
        writer,
        sheet_name='Obs_vs_Est',
        index=True
    )

    metricas_obs_est.to_excel(
        writer,
        sheet_name='Metrics_Obs_Est',
        index=False
    )


print(
    f"\nFile exported: {SALIDA}"
)

In [ ]:
# ============================================================
# Apply Trained Box-Cox Model to New Data
# ============================================================
# Uses a previously calibrated Box-Cox multiple regression model
# to estimate the response variable for new observations and
# export prediction results to Excel.
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm

# ── CONFIGURATION ───────────────────────────────────────────
RUTA_NUEVOS = str(REPO_ROOT / 'data' / 'reflectance' / 'processed' / 'Reflectance_2013_2018_grouped.xlsx')
HOJA_NUEVOS = '2015-2016'
SALIDA_PRED = str(PREDICTIONS_DIR / f'Predictions_new_points_{HOJA}.xlsx')

# ── 1. LOAD NEW DATA ────────────────────────────────────────
df_nuevos = pd.read_excel(RUTA_NUEVOS, sheet_name=HOJA_NUEVOS)
print(f"Sheet: {HOJA_NUEVOS} | Dimensions: {df_nuevos.shape}")
print("Columns:", df_nuevos.columns.tolist())

# Verify all required band columns are present
missing = [c for c in cols_x if c not in df_nuevos.columns]
if missing:
    raise ValueError(f"Missing columns in new data: {missing}")

# ── 2. CLEANING ─────────────────────────────────────────────
datos_nuevos = (
    df_nuevos[cols_x]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .reset_index(drop=True)
    .astype(float)
)

# Keep any extra columns from the original file (ID, point name, etc.)
cols_extra = [c for c in df_nuevos.columns if c not in cols_x]
if cols_extra:
    info_extra = df_nuevos.loc[datos_nuevos.index, cols_extra].reset_index(drop=True)
else:
    info_extra = None

print(f"\nValid rows for prediction: {len(datos_nuevos)}")

# ── 3. APPLY MODEL (no re-estimation) ───────────────────────
X_nuevos       = datos_nuevos[cols_x].values
X_nuevos_const = sm.add_constant(X_nuevos, has_constant='add')

# Prediction on W scale
W_pred_nuevos = np.asarray(modelo.predict(X_nuevos_const))

# Back-transform to original Y scale
Y_pred_nuevos = boxcox_inverse(W_pred_nuevos, lambda_opt, LAMBDA2, K2)

# Confidence and prediction intervals on Y scale
pred_obj   = modelo.get_prediction(X_nuevos_const)
pred_frame = pred_obj.summary_frame(alpha=ALPHA)

Y_CI_lower = boxcox_inverse(pred_frame['mean_ci_lower'].values, lambda_opt, LAMBDA2, K2)
Y_CI_upper = boxcox_inverse(pred_frame['mean_ci_upper'].values, lambda_opt, LAMBDA2, K2)
Y_PI_lower = boxcox_inverse(pred_frame['obs_ci_lower'].values,  lambda_opt, LAMBDA2, K2)
Y_PI_upper = boxcox_inverse(pred_frame['obs_ci_upper'].values,  lambda_opt, LAMBDA2, K2)

# ── 4. BUILD RESULTS TABLE ───────────────────────────────────
tabla_pred_nuevos = datos_nuevos.copy()
tabla_pred_nuevos[f'{COL_Y}_estimated'] = Y_pred_nuevos
tabla_pred_nuevos['CI_lower_95%']       = Y_CI_lower
tabla_pred_nuevos['CI_upper_95%']       = Y_CI_upper
tabla_pred_nuevos['PI_lower_95%']       = Y_PI_lower
tabla_pred_nuevos['PI_upper_95%']       = Y_PI_upper

# Prepend extra columns if they existed (ID, point name, etc.)
if info_extra is not None:
    tabla_pred_nuevos = pd.concat([info_extra, tabla_pred_nuevos], axis=1)

# ── 5. CONSOLE OUTPUT ────────────────────────────────────────
print("\n" + "=" * 65)
print(f"PREDICTIONS  –  {COL_Y}  (model: sheet {HOJA})")
print("=" * 65)
print(f"  Lambda1 : {lambda_opt:.4f}")
print(f"  Lambda2 : {LAMBDA2}")
print(f"  K2      : {K2:.6f}")
print(f"  Model   : W = {modelo.params[0]:.6f} + "
      + " + ".join([f"{p:.6f}·{v}" for p, v in zip(modelo.params[1:], cols_x)]))
print()
print(tabla_pred_nuevos.round(6).to_string(index=True))

# Report NaN in intervals if any
nans = tabla_pred_nuevos[['CI_lower_95%', 'CI_upper_95%',
                           'PI_lower_95%', 'PI_upper_95%']].isna().sum()
if nans.any():
    print("\nNaN in intervals (base <= 0 when inverting Box-Cox):")
    print(nans[nans > 0])

# ── 6. EXPORT TO EXCEL ───────────────────────────────────────
with pd.ExcelWriter(SALIDA_PRED, engine='openpyxl') as writer:
    tabla_pred_nuevos.to_excel(writer, sheet_name='Predictions', index=True)

print(f"\nFile exported: {SALIDA_PRED}")

In [ ]:
# ============================================================
# Plot Predictions for New Observations
# ============================================================
# Generates visual summaries of estimated values, confidence
# intervals, prediction intervals, uncertainty, and distribution
# comparisons for the new input data.
#
# The modeling period is automatically extracted from
# HOJA_NUEVOS and displayed ABOVE the title of each plot/panel.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm


# ── GRAPH FONT SIZES ─────────────────────────────────────────

plt.rcParams.update({
    'font.size': 15,
    'axes.titlesize': 17,
    'axes.labelsize': 15,
    'xtick.labelsize': 15,
    'ytick.labelsize': 15,
    'legend.fontsize': 15,
})


# ============================================================
# AUTOMATIC PERIOD
# ============================================================
#
# Examples:
# Cha-a_2013-2014 → 2013–2014
# Cha-a_2015-2016 → 2015–2016
# Cha-a_2017-2018 → 2017–2018
#
# The period will automatically change when HOJA_NUEVOS changes.
# ============================================================

PERIODO = HOJA_NUEVOS.split('_')[-1].replace('-', '–')

print(f"Period used in figures: {PERIODO}")


# ============================================================
# PREPARATION
# ============================================================

col_est = f'{COL_Y}_estimated'

n_nuevos = len(tabla_pred_nuevos)


# X-axis label:
# use first extra column as ID if available,
# otherwise point number

if info_extra is not None and len(info_extra.columns) > 0:

    id_col = info_extra.columns[0]

    labels = (
        tabla_pred_nuevos[id_col]
        .astype(str)
        .tolist()
    )

    xlabel_txt = id_col

else:

    labels = [
        f"P{i+1}"
        for i in range(n_nuevos)
    ]

    xlabel_txt = "Point"


x_pos = np.arange(n_nuevos)


# Mean of observed training values

media_y = np.mean(y)


# ============================================================
# COLOR PALETTE PROPORTIONAL TO ESTIMATED VALUE
# ============================================================

norm = plt.Normalize(
    tabla_pred_nuevos[col_est].min(),
    tabla_pred_nuevos[col_est].max()
)


colors = cm.RdYlGn(
    norm(
        tabla_pred_nuevos[col_est].values
    )
)


# ============================================================
# HALF-WIDTHS FOR CI AND PI ERROR BARS
# ============================================================

ci_err = np.array([

    tabla_pred_nuevos[col_est]
    - tabla_pred_nuevos['CI_lower_95%'],

    tabla_pred_nuevos['CI_upper_95%']
    - tabla_pred_nuevos[col_est]

])


pi_err = np.array([

    tabla_pred_nuevos[col_est]
    - tabla_pred_nuevos['PI_lower_95%'],

    tabla_pred_nuevos['PI_upper_95%']
    - tabla_pred_nuevos[col_est]

])


# Interval widths

amplitud_pi = (
    tabla_pred_nuevos['PI_upper_95%']
    - tabla_pred_nuevos['PI_lower_95%']
)


amplitud_ci = (
    tabla_pred_nuevos['CI_upper_95%']
    - tabla_pred_nuevos['CI_lower_95%']
)


# ============================================================
# HELPER FUNCTION TO SAVE EACH FIGURE
# ============================================================

def save_fig(fig, name):

    ruta = (
        f'{CARPETA_SALIDA}/'
        f'{name}_{COL_Y}_{HOJA_NUEVOS}.png'
    )

    fig.savefig(
        ruta,
        dpi=150,
        bbox_inches='tight'
    )

    print(f"Saved: {ruta}")

    plt.show()

    plt.close(fig)


# ============================================================
# PLOT 1
# Estimated value per point
# ============================================================

fig, ax = plt.subplots(
    figsize=(
        max(10, n_nuevos * 0.7),
        6
    )
)


bars = ax.bar(
    x_pos,
    tabla_pred_nuevos[col_est],
    color=colors,
    edgecolor='gray',
    linewidth=0.5
)


# Training mean

ax.axhline(
    media_y,
    ls='--',
    color='steelblue',
    lw=1.5,
    label=f'Training mean ({media_y:.3f})'
)


# Values above bars

for bar, val in zip(
    bars,
    tabla_pred_nuevos[col_est]
):

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + ax.get_ylim()[1] * 0.01,
        f'{val:.2f}',
        ha='center',
        va='bottom',
        fontsize=15
    )


ax.set_xticks(x_pos)


ax.set_xticklabels(
    labels,
    rotation=45,
    ha='right',
    fontsize=16
)


ax.set_ylabel(
    f'{COL_Y} estimated'
)


# ── PERIOD ABOVE TITLE ───────────────────────────────────────

ax.set_title(
    f'{PERIODO}\n'
    f'Estimated {COL_Y} per point\n'
    f'(λ1={lambda_opt:.4f}, λ2={LAMBDA2}, K2={K2:.4f})'
)


ax.legend()


ax.grid(
    axis='y',
    alpha=0.3
)


# Color bar

sm_cb = plt.cm.ScalarMappable(
    cmap='RdYlGn',
    norm=norm
)

sm_cb.set_array([])


fig.colorbar(
    sm_cb,
    ax=ax,
    orientation='vertical',
    fraction=0.03,
    pad=0.02,
    label=f'{COL_Y} estimated'
)


fig.tight_layout()


save_fig(
    fig,
    '07_Estimated_per_point'
)


# ============================================================
# PLOT 2
# Estimated values with CI and PI
# ============================================================

fig, ax = plt.subplots(
    figsize=(
        max(10, n_nuevos * 0.7),
        6
    )
)


# Prediction intervals

ax.errorbar(
    x_pos,
    tabla_pred_nuevos[col_est],
    yerr=pi_err,
    fmt='none',
    color='lightcoral',
    linewidth=6,
    alpha=0.5,
    label='PI 95%'
)


# Confidence intervals

ax.errorbar(
    x_pos,
    tabla_pred_nuevos[col_est],
    yerr=ci_err,
    fmt='none',
    color='steelblue',
    linewidth=3,
    alpha=0.8,
    label='CI 95%'
)


# Estimated values

ax.scatter(
    x_pos,
    tabla_pred_nuevos[col_est],
    color='black',
    zorder=5,
    s=50,
    label='Estimated'
)


# Training mean

ax.axhline(
    media_y,
    ls='--',
    color='steelblue',
    lw=1,
    label=f'Training mean ({media_y:.3f})'
)


ax.set_xticks(x_pos)


ax.set_xticklabels(
    labels,
    rotation=45,
    ha='right',
    fontsize=16
)


ax.set_ylabel(
    f'{COL_Y}'
)


# ── PERIOD ABOVE TITLE ───────────────────────────────────────

ax.set_title(
    f'{PERIODO}\n'
    f'Estimated {COL_Y} with 95% Confidence and Prediction Intervals'
)


ax.legend(
    loc='upper right'
)


ax.grid(
    axis='y',
    alpha=0.3
)


fig.tight_layout()


save_fig(
    fig,
    '08_Estimated_CI_PI'
)


# ============================================================
# PLOT 3
# Interval width — uncertainty per point
# ============================================================

fig, ax = plt.subplots(
    figsize=(
        max(10, n_nuevos * 0.7),
        6
    )
)


ax.bar(
    x_pos - 0.2,
    amplitud_pi,
    width=0.4,
    color='lightcoral',
    edgecolor='gray',
    label='PI 95% width'
)


ax.bar(
    x_pos + 0.2,
    amplitud_ci,
    width=0.4,
    color='steelblue',
    edgecolor='gray',
    label='CI 95% width'
)


ax.set_xticks(x_pos)


ax.set_xticklabels(
    labels,
    rotation=45,
    ha='right',
    fontsize=16
)


ax.set_ylabel(
    'Interval width'
)


# ── PERIOD ABOVE TITLE ───────────────────────────────────────

ax.set_title(
    f'{PERIODO}\n'
    f'Uncertainty per point\n'
    f'(narrower interval = more precise prediction)'
)


ax.legend()


ax.grid(
    axis='y',
    alpha=0.3
)


fig.tight_layout()


save_fig(
    fig,
    '09_Uncertainty_per_point'
)


# ============================================================
# PLOT 4
# Distribution comparison
#
# Left  = Histogram of estimated values
# Right = Training observations vs estimated new points
#
# IMPORTANT:
# The period appears above the title of EACH panel.
# This allows panels to be combined later with other
# parameters and periods without losing identification.
# ============================================================

fig, axes2 = plt.subplots(
    1,
    2,
    figsize=(14, 6)
)


# ============================================================
# PLOT 4a
# Histogram of estimated values
# ============================================================

ax = axes2[0]


ax.hist(
    tabla_pred_nuevos[col_est],
    bins='auto',
    color='steelblue',
    edgecolor='white',
    alpha=0.8
)


# Training mean

ax.axvline(
    media_y,
    ls='--',
    color='red',
    lw=1.5,
    label=f'Training mean ({media_y:.3f})'
)


# Training range

ax.axvspan(
    y.min(),
    y.max(),
    alpha=0.1,
    color='green',
    label=(
        f'Training range '
        f'[{y.min():.2f}, {y.max():.2f}]'
    )
)


ax.set_xlabel(
    f'{COL_Y} estimated'
)


ax.set_ylabel(
    'Frequency'
)


# ── PERIOD ABOVE HISTOGRAM TITLE ─────────────────────────────

ax.set_title(
    f'{PERIODO}\n'
    f'Histogram'
)


ax.legend(
    fontsize=14
)


ax.grid(
    alpha=0.3
)


# ============================================================
# PLOT 4b
# Boxplot: training vs new points
# ============================================================

ax = axes2[1]


ax.boxplot(

    [
        y,
        tabla_pred_nuevos[col_est].values
    ],

    labels=[
        'Training\n(observed)',
        'New points\n(estimated)'
    ],

    patch_artist=True,

    boxprops=dict(
        facecolor='lightblue'
    ),

    medianprops=dict(
        color='red',
        linewidth=2
    )
)


ax.set_ylabel(
    f'{COL_Y}'
)


# ── PERIOD ABOVE BOXPLOT TITLE ───────────────────────────────

ax.set_title(
    f'{PERIODO}\n'
    f'Distribution comparison:\n'
    f'training vs new points'
)


ax.grid(
    axis='y',
    alpha=0.3
)


# Adjust layout
fig.tight_layout()


save_fig(
    fig,
    '10_Distribution_comparison'
)


# ============================================================
# FINAL MESSAGE
# ============================================================

print(
    f"\nAll plots saved individually in: "
    f"{CARPETA_SALIDA}"
)

print(
    f"Period included above each figure/panel title: "
    f"{PERIODO}"
)